In [3]:
%pip install python-dotenv langchain-upstage


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain langchain-community langchain-text-splitters langchain-pinecone docx2txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

ai_message = llm.invoke("한국에 대해 3줄로 설명해줘")

ai_message.content

In [ ]:
# docx파일 load, split and saving in the vector storage

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./접구선통기술기준_20251128.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [7]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_gl = 'groundandlinelaw-index'
database1 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_gl)

In [8]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./방송통신설비기술기준_20240719.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_bc = 'broadcastandcomlaw-index'
database2 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_bc)

In [ ]:
%pip install langchain-pinecone

In [8]:
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name_gl = 'groundandlinelaw-index'
index_name_bc = 'broadcastandcomlaw-index'

groundline_db = PineconeVectorStore(index_name=index_name_gl, embedding=embedding)
broadcom_db = PineconeVectorStore(index_name=index_name_bc, embedding=embedding)

In [21]:
query = '통신설비의 접지저항은 얼마입니까?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
docs1 = groundline_db.similarity_search_with_score(query, k=5)
docs2 = broadcom_db.similarity_search_with_score(query, k=5)

all_docs = docs1 + docs2

all_docs = sorted(all_docs, key=lambda x:x[1],reverse=True)

In [ ]:
all_docs

In [ ]:
# 접구선통기술기준 파이콘에서 삭제후 저장, 메타데이터(법률/몇조/몇항)추가해서 저장

import os
import re

from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("GROUNDLINE_INDEX")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

# -----------------------------
# 1️⃣ Pinecone 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

# -----------------------------
# 2️⃣ 기존 데이터 삭제
# -----------------------------
print("기존 Pinecone 데이터 삭제 중...")

index.delete(delete_all=True, namespace=NAMESPACE)

print("삭제 완료")

# -----------------------------
# 3️⃣ docx 파일 로드
# -----------------------------
loader = Docx2txtLoader("./접구선통기술기준_20251128.docx")
documents = loader.load()

# -----------------------------
# 4️⃣ 문서 분할
# -----------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

# -----------------------------
# 5️⃣ 조문 metadata 추출 함수
# -----------------------------
def extract_article_clause(text):

    article_match = re.search(r"(제\s*\d+\s*조)", text)
    clause_match = re.search(r"(제\s*\d+\s*항)", text)

    article = article_match.group(1).replace(" ", "") if article_match else ""
    clause = clause_match.group(1).replace(" ", "") if clause_match else ""

    return article, clause


# -----------------------------
# 6️⃣ metadata 추가
# -----------------------------
for doc in split_docs:

    article, clause = extract_article_clause(doc.page_content)

    doc.metadata["regulation_name"] = "접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준"
    doc.metadata["article"] = article
    doc.metadata["clause"] = clause
    doc.metadata["source"] = "./접구선통기술기준_20251128.docx"

print("metadata 추가 완료")

# -----------------------------
# 7️⃣ 임베딩 모델
# -----------------------------
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
)

# -----------------------------
# 8️⃣ Pinecone VectorStore
# -----------------------------
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE
)

# -----------------------------
# 9️⃣ Pinecone 저장
# -----------------------------
vectorstore.add_documents(split_docs)

print("새로운 조문 데이터 업로드 완료")

기존 Pinecone 데이터 삭제 중...
삭제 완료
metadata 추가 완료
새로운 조문 데이터 업로드 완료


In [ ]:
# 방송통신설비기술기준 파이콘에서 삭제후 저장, 메타데이터(법률/몇조/몇항)추가해서 저장

import os
import re

from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_upstage import UpstageEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("BROADCOM_INDEX")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

# -----------------------------
# 1️⃣ Pinecone 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

# -----------------------------
# 2️⃣ 기존 데이터 삭제
# -----------------------------
print("기존 Pinecone 데이터 삭제 중...")

index.delete(delete_all=True, namespace=NAMESPACE)

print("삭제 완료")

# -----------------------------
# 3️⃣ docx 파일 로드
# -----------------------------
loader = Docx2txtLoader("./방송통신설비기술기준_20240719.docx")
documents = loader.load()

# -----------------------------
# 4️⃣ 문서 분할
# -----------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

# -----------------------------
# 5️⃣ 조문 metadata 추출 함수
# -----------------------------
def extract_article_clause(text):

    article_match = re.search(r"(제\s*\d+\s*조)", text)
    clause_match = re.search(r"(제\s*\d+\s*항)", text)

    article = article_match.group(1).replace(" ", "") if article_match else ""
    clause = clause_match.group(1).replace(" ", "") if clause_match else ""

    return article, clause


# -----------------------------
# 6️⃣ metadata 추가
# -----------------------------
for doc in split_docs:

    article, clause = extract_article_clause(doc.page_content)

    doc.metadata["regulation_name"] = "방송통신설비의 기술기준에 관한 규정"
    doc.metadata["article"] = article
    doc.metadata["clause"] = clause
    doc.metadata["source"] = "./방송통신설비기술기준_20240719.docx"

print("metadata 추가 완료")

# -----------------------------
# 7️⃣ 임베딩 모델
# -----------------------------
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
)

# -----------------------------
# 8️⃣ Pinecone VectorStore
# -----------------------------
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE
)

# -----------------------------
# 9️⃣ Pinecone 저장
# -----------------------------
vectorstore.add_documents(split_docs)

print("새로운 조문 데이터 업로드 완료")